# Extract Fig 4d V1 mean flash PSTH (Siegle et al. 2021)

Mirrors the loop at the bottom of `Figure3/get_flash_PSTH.py` in the paper repo.

Pipeline per session (Brain Observatory 1.1 only):
1. Load NWB via `EcephysProjectCache.get_session_data(session_id)`
2. Get the `flashes` stimulus presentation table
3. Restrict to VISp units passing Siegle's Fig 4 QC filter:
   `on_screen_rf < 0.01 & time_to_first_spike_fl < 0.1 & area_rf < 2500 & firing_rate_dg > 0.1 & snr > 1`
4. For each unit, compute trial-averaged PSTH over 0–120 ms post-flash in 1 ms bins
   via `session.presentationwise_spike_counts(...)`
5. Stack all units across sessions, mean across units

Then post-process exactly like the published code:
- baseline-subtract using mean of first 20 ms (this is the *post-flash* baseline they use)
- divide by `bin_size_s` to convert counts/bin to spikes/s
- Gaussian smooth with sigma = 2 bins (= 2 ms)

Output: `data/fig4d_v1_flash_psth.npz` with arrays `t_ms`, `psth_hz`, `psth_raw_hz`, `n_units`, `n_sessions`, `session_ids`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d

from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache

CACHE_DIR = Path('/Users/pmccarthy/Documents/experimental_data/allen_visual_neuropixels_longwindow_5ms_bins')
OUT_PATH  = Path('../data/fig4d_v1_flash_psth.npz')
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# PSTH window + bin size matching get_flash_PSTH.py:
BIN_S   = 0.001                                  # 1 ms bins
T_PRE   = 0.0
T_POST  = 0.120                                  # 0–120 ms post-flash
BASELINE_BINS = 20                               # first 20 ms used as baseline
SMOOTH_SIGMA  = 2                                # gaussian_filter1d(psth, 2)

bin_edges = np.arange(T_PRE, T_POST + BIN_S/2, BIN_S)  # length 121
t_ms = ((bin_edges[:-1] + bin_edges[1:]) / 2) * 1000   # bin centers, length 120
print(f'PSTH bins: {len(t_ms)} (0–{T_POST*1000:.0f} ms in {BIN_S*1000:.0f} ms steps)')

cache = EcephysProjectCache.from_warehouse(manifest=str(CACHE_DIR / 'manifest.json'))
sessions_table = cache.get_session_table()
bo_sessions = sessions_table[sessions_table.session_type == 'brain_observatory_1.1'].index.tolist()
print(f'{len(bo_sessions)} brain_observatory_1.1 sessions')

In [ ]:
# Apply Siegle's Fig 4 QC filter using the cache's analysis metrics.
# Use filter_by_validity=False + permissive thresholds so we can apply our own.
metrics = cache.get_unit_analysis_metrics_by_session_type(
    'brain_observatory_1.1',
    filter_by_validity=False,
    amplitude_cutoff_maximum=np.inf,
    presence_ratio_minimum=-np.inf,
    isi_violations_maximum=np.inf,
)

qc_mask = (
    (metrics.ecephys_structure_acronym == 'VISp')
    & (metrics.on_screen_rf < 0.01)
    & (metrics.time_to_first_spike_fl < 0.1)
    & (metrics.area_rf < 2500)
    & (metrics.firing_rate_dg > 0.1)
    & (metrics.snr > 1)
)
good_unit_ids = set(metrics.index[qc_mask].astype(int).tolist())
print(f'{len(good_unit_ids):,} VISp units pass the Fig 4 QC filter')

In [ ]:
# Compute per-unit trial-averaged PSTH, accumulate across sessions.
all_unit_psths = []   # list of (n_units_in_session, n_bins) trial-averaged arrays
used_session_ids = []

for i, sid in enumerate(bo_sessions, 1):
    print(f'[{i}/{len(bo_sessions)}] session {sid} ...', end=' ', flush=True)
    try:
        session = cache.get_session_data(sid)
        flashes = session.get_stimulus_table('flashes')
        if len(flashes) == 0:
            print('no flash trials, skip'); continue

        # Only VISp units in this session that pass QC.
        sess_units = session.units.index.values.astype(int)
        unit_ids = np.array([u for u in sess_units if u in good_unit_ids])
        if len(unit_ids) == 0:
            print('0 V1 QC units, skip'); continue

        # presentationwise_spike_counts returns an xarray with dims
        # (stimulus_presentation_id, time_relative_to_stimulus_onset, unit_id).
        counts = session.presentationwise_spike_counts(
            bin_edges=bin_edges,
            stimulus_presentation_ids=flashes.index.values,
            unit_ids=unit_ids,
        )
        # shape (n_trials, n_bins, n_units) -> mean over trials -> (n_units, n_bins)
        unit_psth_hz = counts.mean(dim='stimulus_presentation_id').transpose('unit_id', 'time_relative_to_stimulus_onset').values / BIN_S
        all_unit_psths.append(unit_psth_hz)
        used_session_ids.append(sid)
        print(f'{len(flashes)} flashes × {len(unit_ids)} units')
    except Exception as e:
        print(f'ERROR: {e}')

all_unit_psths = np.concatenate(all_unit_psths, axis=0)
print(f'\nstacked: {all_unit_psths.shape[0]:,} V1 units across {len(used_session_ids)} sessions')

In [ ]:
# Mean across units, then post-process per get_flash_PSTH.py.
psth_raw_hz = all_unit_psths.mean(axis=0)                   # (n_bins,) spikes/s
psth = psth_raw_hz - psth_raw_hz[:BASELINE_BINS].mean()     # baseline subtract
psth = gaussian_filter1d(psth, SMOOTH_SIGMA)                # smooth

np.savez(
    OUT_PATH,
    t_ms=t_ms,
    psth_hz=psth,
    psth_raw_hz=psth_raw_hz,
    n_units=all_unit_psths.shape[0],
    n_sessions=len(used_session_ids),
    session_ids=np.array(used_session_ids),
    bin_size_ms=BIN_S * 1000,
    baseline_bins=BASELINE_BINS,
    smooth_sigma_ms=SMOOTH_SIGMA,
)
print(f'Saved → {OUT_PATH.resolve()}')

In [ ]:
# Quick look
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(t_ms, psth, color='C0', lw=2, label=f'V1 (n={all_unit_psths.shape[0]} units)')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('Time after flash onset (ms)')
ax.set_ylabel('Mean firing rate, baseline-subtracted (sp/s)')
ax.set_title('Fig 4d: V1 flash response (Siegle et al. 2021)')
for s in ('top','right'): ax.spines[s].set_visible(False)
ax.legend()
plt.tight_layout()
plt.savefig('../data/fig4d_v1_flash_psth.png', dpi=130)
plt.show()